In [0]:
from pyspark.sql import functions as F

# Bronze Volume Path
bronze_path = "/Volumes/tt_hc_adb_ws/bronze/bronze_volume"

# Reading Hospital A providers data
df_hosa = spark.read.parquet(f"{bronze_path}/hosa/providers")

# Reading Hospital B providers data
df_hosb = spark.read.parquet(f"{bronze_path}/hosb/providers")

# Union both DataFrames
df_merged = df_hosa.unionByName(df_hosb)

# Create Temp View
df_merged.createOrReplaceTempView("providers")

display(df_merged)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tt_hc_adb_ws.silver.providers (
ProviderID string,
FirstName string,
LastName string,
Specialization string,
DeptID string,
NPI long,
datasource string,
is_quarantined boolean
)
USING DELTA;

In [0]:
%sql
truncate table tt_hc_adb_ws.silver.providers

In [0]:
%sql 
insert into tt_hc_adb_ws.silver.providers
select 
distinct
ProviderID,
FirstName,
LastName,
Specialization,
DeptID,
cast(NPI as INT) NPI,
datasource,
    CASE 
        WHEN ProviderID IS NULL OR DeptID IS NULL THEN TRUE
        ELSE FALSE
    END AS is_quarantined
from providers